# Stage 3 — Itinerary Builder (K-Means++ + TSP)
**[STUDENT VERSION — fill in the blanks]**

Stage 3 không còn trả lời câu hỏi **“nên đi đâu?”**, mà chuyển sang câu hỏi **“chia các điểm đã chọn theo ngày và đi theo thứ tự nào?”**.

## Pipeline của stage

```text
Top-K từ Stage 2 + tọa độ từ Stage 1 + số ngày cấu hình
                         ↓
               danh sách điểm đã chọn
                         ↓
       K-Means++: chia điểm thành K cụm/ngày
                         ↓
  Nearest Neighbor multi-start: xếp thứ tự trong từng ngày
                         ↓
 Haversine: ước lượng các chặng và tổng quãng đường chim bay
                         ↓
             itinerary dạng CSV và JSON
```

Bài toán được **phân rã thành hai heuristic**:
- Stage 3a — K-Means++: gom các điểm gần nhau vào cùng ngày để hạn chế di chuyển xa giữa các điểm trong ngày.
- Stage 3b — Nearest Neighbor multi-start: tìm một thứ tự đi hợp lý trong từng ngày. Đây là lời giải xấp xỉ, không bảo đảm ngắn nhất tuyệt đối.

## Mục tiêu học tập

Sau notebook này, bạn có thể:
1. phân biệt bài toán **recommendation** với bài toán **lập lịch trình**;
2. giải thích mục tiêu, bước gán cụm và cập nhật tâm cụm của K-Means;
3. giải thích vì sao K-Means++ chọn tâm ban đầu theo xác suất tỉ lệ với bình phương khoảng cách;
4. triển khai và đánh giá heuristic Nearest Neighbor multi-start;
5. nhận biết sự khác nhau giữa khoảng cách trên tọa độ, khoảng cách đường chim bay và khoảng cách đường bộ;
6. nêu được giới hạn của cách tách bài toán thành “chia ngày trước, xếp tuyến sau”.

> 💡 Các cell có `# TODO` là phần học viên cần hoàn thiện. Notebook giữ thuật toán đủ nhỏ để có thể đọc và kiểm tra từng bước.

In [126]:
import csv, json, math, random
from pathlib import Path
import pandas as pd


In [127]:
BASE_DIR       = Path('.')
STAGE2_CSV     = BASE_DIR / '../input/stage2/stage2_results.csv'
DATASET_CSV    = BASE_DIR / '../input/stage1/stage1_dataset.csv'
CONFIG_CSV     = BASE_DIR / '../input/stage3/stage3_config.csv'
ITINERARY_CSV  = BASE_DIR / '../input/stage3/stage3_itinerary.csv'
ITINERARY_JSON = BASE_DIR / '../input/stage3/stage3_itinerary.json'

def read_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

stage2_rows = read_csv(STAGE2_CSV)
dataset     = read_csv(DATASET_CSV)
config_rows = read_csv(CONFIG_CSV)
print(f'Stage 2 rows: {len(stage2_rows)}')
print(f'Dataset: {len(dataset)}')


Stage 2 rows: 10
Dataset: 42


In [128]:
# Distance helpers (đã có sẵn, không cần sửa)
def euclidean(a, b):
    """Euclidean distance on (lat,lng) — dùng cho K-Means và TSP so sánh."""
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def haversine_km(a, b):
    """Khoảng cách km thực tế (đường chim bay) — dùng để hiển thị output."""
    R=6371; dlat=math.radians(b[0]-a[0]); dlng=math.radians(b[1]-a[1])
    x=math.sin(dlat/2)**2+math.cos(math.radians(a[0]))*math.cos(math.radians(b[0]))*math.sin(dlng/2)**2
    return R*2*math.atan2(math.sqrt(x),math.sqrt(1-x))


## 🔧 TODO 1 — K-Means++ Initialization

### 1. K-Means đang tối ưu điều gì?

Mỗi điểm đến là một điểm `x_i = (lat, lng)`. Với `K = số ngày`, K-Means tìm cách gán mỗi điểm vào một cụm sao cho các điểm trong cùng cụm gần tâm cụm của chúng. Hàm mục tiêu thường viết:

`J = Σ_i ||x_i - μ_(c_i)||²`

Trong đó `c_i` là cụm của điểm `i`, còn `μ_(c_i)` là centroid của cụm đó. Giá trị `J` càng nhỏ thì các cụm càng “gọn” theo thước đo đang dùng.

Sau khi khởi tạo centroid, K-Means lặp hai bước:
1. **Assign:** gán mỗi điểm vào centroid gần nhất.
2. **Update:** thay mỗi centroid bằng trung bình tọa độ của các điểm vừa được gán vào cụm.

Lặp đến khi centroid không đổi hoặc đạt `n_iter`. Lưu ý: K-Means chỉ hội tụ tới một nghiệm cục bộ; kết quả phụ thuộc đáng kể vào các centroid ban đầu.

### 2. Vì sao dùng K-Means++?

K-Means chuẩn thường khởi tạo ngẫu nhiên nên nhiều centroid có thể nằm sát nhau và dẫn tới cụm kém. K-Means++ phân tán các centroid ban đầu tốt hơn:
- chọn centroid đầu tiên;
- với mỗi điểm `x`, tính `D(x)` là khoảng cách tới centroid gần nhất đã có;
- chọn centroid tiếp theo theo xác suất `P(x) = D(x)² / Σ_z D(z)²`.

Bình phương khoảng cách làm các điểm đang ở xa mọi centroid có cơ hội được chọn cao hơn. Đây chỉ là **khởi tạo**; sau đó vẫn phải chạy vòng lặp Assign → Update của K-Means.

### 3. Tính ngẫu nhiên và seed

Phép “roulette wheel” dùng số ngẫu nhiên, vì vậy hai lần chạy có thể cho cách chia ngày khác nhau. Khi cần tái lập kết quả để học hoặc kiểm thử, có thể đặt một seed cố định cho module `random` trước khi gọi thuật toán. Seed chỉ làm thí nghiệm lặp lại được, không tự làm nghiệm tốt hơn; khi đánh giá chất lượng nên thử nhiều seed.

### 4. Độ phức tạp của phần phân cụm

Gọi `n` là số điểm, `k` là số ngày và `I` là số vòng lặp:
- mỗi bước Assign so sánh `n` điểm với `k` centroid: `O(nk)`;
- bước Update duyệt lại các thành viên cụm: xấp xỉ `O(n)`;
- phần lặp K-Means vì thế khoảng `O(I·n·k)`;
- cách viết khởi tạo bên dưới tính lại khoảng cách tới mọi centroid đã có, nên tổng chi phí khởi tạo trực tiếp khoảng `O(n·k²)`.

Bộ nhớ chính là danh sách cụm, centroid và khoảng cách tạm, cỡ `O(n + k)`.

### 5. Euclidean trên `(lat, lng)` là một xấp xỉ

Notebook dùng Euclidean trên hai giá trị độ vĩ/kinh để gán cụm. Cách này đơn giản và có thể dùng như xấp xỉ trên một vùng không quá rộng, nhưng:
- một độ kinh tuyến không tương ứng cùng số km ở mọi vĩ độ;
- bề mặt Trái Đất không phải mặt phẳng;
- khoảng cách địa lý chưa phản ánh đường sá thực tế.

`haversine_km()` cho khoảng cách cung lớn theo km, tức **đường chim bay trên mặt cầu**, không phải quãng đường lái xe. Trong notebook này, Euclidean được dùng khi so sánh điểm cho K-Means và bước chọn điểm gần nhất; Haversine được dùng để chấm tổng chiều dài tuyến và hiển thị các chặng. Hai thước đo có thể cho thứ tự hơi khác nhau.

Điền phần khởi tạo K-Means++ vào hàm `kmeans_plus_plus()` bên dưới.


In [129]:
def kmeans_plus_plus(points, k, n_iter=60):
    """
    K-Means++ clustering trên (lat, lng).
    points: list of (lat, lng)
    k: số ngày = số cụm
    Returns: list of k lists of indices
    """
    if len(points) <= k:
        return [[i] for i in range(len(points))]

    # TODO 1a: K-Means++ init
    centroids = [points[0]]  # centroid đầu tiên = điểm đầu tiên

    while len(centroids) < k:
        # Tính d² từ mỗi điểm đến centroid gần nhất
        d2    = [min(euclidean(p, c) ** 2 for c in centroids) for p in points]  # ← [min(euclidean(p,c)**2 for c in centroids) for p in points]
        total = sum(d2)  # ← sum(d2)

        # Roulette wheel selection theo xác suất tỉ lệ với d²
        r      = random.uniform(0, total)
        cumsum = 0.0
        for i, d in enumerate(d2):
            cumsum += d
            if cumsum >= r:
                centroids.append(points[i])
                break
        else:
            centroids.append(points[-1])

    # Assign + Recompute (đã có sẵn)
    clusters = None
    for _ in range(n_iter):
        clusters = [[] for _ in range(k)]
        for i, p in enumerate(points):
            nearest = min(range(k), key=lambda ci: euclidean(p, centroids[ci]))
            clusters[nearest].append(i)
        new_c = []
        for ci in range(k):
            if clusters[ci]:
                new_c.append((sum(points[i][0] for i in clusters[ci])/len(clusters[ci]),
                              sum(points[i][1] for i in clusters[ci])/len(clusters[ci])))
            else:
                new_c.append(centroids[ci])
        if new_c == centroids: break
        centroids = new_c

    return clusters

print('kmeans_plus_plus() defined.')


kmeans_plus_plus() defined.


## 🔧 TODO 2 — Nearest Neighbor (multi-start)

### Greedy Nearest Neighbor

Với một điểm xuất phát, heuristic tham lam thực hiện:
1. đánh dấu điểm hiện tại đã thăm;
2. chọn điểm **chưa thăm gần nhất**;
3. di chuyển tới đó và lặp lại cho đến khi mọi điểm đã xuất hiện trong route.

Lựa chọn tốt nhất ở bước hiện tại chưa chắc tạo ra tuyến tốt nhất toàn cục. Vì thế Nearest Neighbor nhanh và dễ hiểu nhưng không phải exact solver.

### Vì sao cần multi-start?

Nearest Neighbor phụ thuộc mạnh vào điểm bắt đầu. `multi-start` chạy heuristic từ **mọi** điểm xuất phát, tính tổng quãng đường của từng route rồi giữ route ngắn nhất đã tìm được. Cách này thường tốt hơn một lần chạy, nhưng vẫn không bảo đảm nghiệm tối ưu.

### Đây là đường Hamilton mở, không phải tour TSP đóng

Code chỉ cộng các chặng giữa hai điểm liên tiếp, tổng cộng `n - 1` chặng. Route **không quay lại điểm xuất phát**. Vì vậy, chính xác hơn đây là bài toán tìm một **đường Hamilton mở** bằng heuristic Nearest Neighbor. TSP cổ điển thường yêu cầu một chu trình khép kín và phải cộng thêm chặng từ điểm cuối về điểm đầu.

### Độ phức tạp

Với `n` điểm trong một ngày:
- một lần Nearest Neighbor cần khoảng `O(n²)` phép so sánh;
- thử cả `n` điểm xuất phát đưa multi-start lên `O(n³)`;
- bộ nhớ phụ trợ khoảng `O(n)` cho route và tập đã thăm.

Chi phí này phù hợp khi mỗi ngày chỉ có ít điểm. Nếu số điểm lớn, cần heuristic mạnh hơn hoặc thư viện tối ưu tuyến chuyên dụng.

Điền phần tìm điểm gần nhất và tính tổng khoảng cách.

In [130]:
def tsp_nearest_neighbor(points):
    """Multi-start Nearest Neighbor TSP. Returns list of indices."""
    n = len(points)
    m = 1 << n
    if n <= 1: return list(range(n))
    dp = [[float('inf') for _ in range(n)] for _ in range(m)]
    trace = [[0 for _ in range(n)] for _ in range(m)]
    for i in range(n):
        dp[1 << i][i] = 0
    for mask in range(m):
        for i in range(n):
            if (mask >> i) & 1 == 0:
                for j in range(n):
                    if (mask >> j) & 1 == 1:
                        if dp[mask | (1 << i)][i] > dp[mask][j] + haversine_km(points[i], points[j]):
                            dp[mask | (1 << i)][i] = dp[mask][j] + haversine_km(points[i], points[j])
                            trace[mask | (1 << i)][i] = j
    best_d = float('inf')
    best_o = []
    last = 0
    for i in range(n):
        if best_d > dp[m-1][i]:
            best_d = dp[m-1][i]
            last = i
    mask = m - 1
    best_o.append(last)
    for _ in range(n-1):
        nxt = trace[mask][last]
        mask ^= (1 << last)
        last = nxt
        best_o.append(last)
    best_o.reverse()
    return best_o
    # print(best_d)    
    # print(best_o)


              
    # best_order, best_dist = None, float('inf')

    # for start in range(n):
    #     visited, order, cur = {start}, [start], start
    #     while len(order) < n:
    #         # TODO 2a: Tìm điểm chưa thăm gần nhất
    #         nxt = min((i for i in range(n) if i not in visited), key = lambda i: euclidean(points[cur], points[i]))  # ← min((i for i in range(n) if i not in visited),
    #                     #        key=lambda i: euclidean(points[cur], points[i]))
    #         visited.add(nxt); order.append(nxt); cur = nxt

    #     # TODO 2b: Tính tổng khoảng cách Haversine của route này
    #     total = sum(haversine_km(points[order[i]], points[order[i+1]]) for i in range(len(order)-1))  # ← sum(haversine_km(points[order[i]], points[order[i+1]])
    #                   #        for i in range(len(order)-1))

    #     if total < best_dist:
    #         best_dist = total; best_order = order
    # print(best_dist)
    # print(best_o)
    # print("haha")
    # return best_order

print('tsp_nearest_neighbor() defined.')


tsp_nearest_neighbor() defined.


## 🔧 TODO 3 — `build_itinerary()` — Kết hợp Stage 3A + 3B

`build_itinerary()` ghép hai bài toán con theo thứ tự:
1. đặt `K = min(n_days, số điểm)` và dùng K-Means++ để tạo các cụm/ngày;
2. với từng cụm, dùng multi-start Nearest Neighbor để tạo thứ tự ghé;
3. tính từng chặng và tổng km đường chim bay bằng Haversine;
4. đóng gói kết quả thành danh sách ngày để xuất CSV/JSON.

Cách phân rã này dễ học và dễ kiểm tra, nhưng **không tối ưu chung toàn bộ chuyến đi**. Một cách chia cụm có tổng sai số K-Means nhỏ chưa chắc tạo ra lịch trình có tổng quãng đường nhỏ nhất.

In [131]:
def build_itinerary(selected_places, n_days):
    """
    K-Means++ phân ngày → TSP tối ưu thứ tự trong ngày.
    Returns: list of day dicts
    """
    k      = min(n_days, len(selected_places))
    coords = [(p['lat'], p['lng']) for p in selected_places]

    # TODO 3a: Gọi K-Means++ để phân cụm
    clusters = kmeans_plus_plus(coords, k)  # ← kmeans_plus_plus(coords, k)

    itinerary = []
    for day_idx, ci in enumerate(clusters):
        if not ci: continue
        day_places = [selected_places[i] for i in ci]
        day_coords = [(p['lat'], p['lng']) for p in day_places]

        # TODO 3b: Gọi TSP để sắp xếp thứ tự
        order  = tsp_nearest_neighbor(day_coords)  # ← tsp_nearest_neighbor(day_coords)
        sorted_day = [day_places[i] for i in order]

        # Tính khoảng cách giữa các stop
        legs, total = [], 0.0
        for i in range(len(sorted_day)-1):
            km = haversine_km((sorted_day[i]['lat'],   sorted_day[i]['lng']),
                              (sorted_day[i+1]['lat'], sorted_day[i+1]['lng']))
            legs.append(round(km,1)); total += km

        itinerary.append({'day':day_idx+1,'stops':sorted_day,
                          'legs_km':legs,'total_km':round(total,1)})
    return itinerary

print('build_itinerary() defined.')


build_itinerary() defined.


In [132]:
# Run + Save (đã có sẵn, không cần sửa)
coord_map = {row['place']:{'lat':float(row['lat']),'lng':float(row['lng']),
             'province':row['province']} for row in dataset}
print(coord_map)
stage2_by_profile = {}
for row in stage2_rows:
    stage2_by_profile.setdefault(row['profile'],[]).append(row)
config = {r['profile_name']:{'n_days':int(r['n_days']),'top_k':int(r['top_k'])} for r in config_rows}

all_results, itinerary_rows = [], []
for profile_name, cfg in config.items():
    n_days, top_k = cfg['n_days'], cfg['top_k']
    s2 = sorted(stage2_by_profile.get(profile_name,[]),key=lambda x:int(x['rank']))[:top_k]
    if not s2: continue
    selected = [{'name':r['place'],'province':coord_map[r['place']]['province'],
                 'lat':coord_map[r['place']]['lat'],'lng':coord_map[r['place']]['lng'],
                 'cosine_score':float(r['cosine_score']),'month_match':r['month_match']}
                for r in s2 if r['place'] in coord_map]
    print(f'\nProfile: {profile_name} | {len(selected)} places | {n_days} days')
    itin = build_itinerary(selected, n_days)
    all_results.append({'profile':profile_name,'n_days':n_days,'itinerary':itin})
    for day in itin:
        for si,stop in enumerate(day['stops']):
            leg = day['legs_km'][si] if si<len(day['legs_km']) else 0.0
            itinerary_rows.append({'profile':profile_name,'day':day['day'],'stop_order':si+1,
                'place':stop['name'],'province':stop['province'],'leg_km':leg,'day_total_km':day['total_km']})
        print(f"  Day {day['day']}: {' → '.join(s['name'] for s in day['stops'])} ({day['total_km']} km)")

headers=['profile','day','stop_order','place','province','leg_km','day_total_km']
with ITINERARY_CSV.open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=headers); w.writeheader(); w.writerows(itinerary_rows)
with ITINERARY_JSON.open('w',encoding='utf-8') as f:
    json.dump(all_results,f,ensure_ascii=False,indent=2)
print(f'\nSaved: {ITINERARY_CSV.name}')
print(f'Saved: {ITINERARY_JSON.name}')


{'phong nha': {'lat': 17.58137, 'lng': 106.28308, 'province': 'Quảng Bình'}, 'hang sơn đoòng': {'lat': 17.44415, 'lng': 106.29358, 'province': 'Quảng Bình'}, 'hang va': {'lat': 17.48947, 'lng': 106.2848, 'province': 'Quảng Bình'}, 'biển nhật lệ': {'lat': 17.492663, 'lng': 106.627266, 'province': 'Quảng Bình'}, 'cồn cát quang phú': {'lat': 17.5302, 'lng': 106.6218, 'province': 'Quảng Bình'}, 'vũng chùa - đảo yến': {'lat': 17.92969, 'lng': 106.5066, 'province': 'Quảng Bình'}, 'quảng bình quan': {'lat': 17.46334, 'lng': 106.62405, 'province': 'Quảng Bình'}, 'tượng đài mẹ suốt': {'lat': 17.46546, 'lng': 106.62714, 'province': 'Quảng Bình'}, 'đền thờ liễu hạnh': {'lat': 17.953538, 'lng': 106.468953, 'province': 'Quảng Bình'}, 'biển đá nhảy': {'lat': 17.662778, 'lng': 106.513056, 'province': 'Quảng Bình'}, 'mộ đại tướng võ nguyên giáp': {'lat': 17.8346, 'lng': 106.368, 'province': 'Quảng Bình'}, 'thành cổ đồng hới': {'lat': 17.4689, 'lng': 106.6223, 'province': 'Quảng Bình '}, 'mũi trèo': {'

## Giới hạn của mô hình và câu hỏi tự kiểm tra

### Giới hạn cần nhớ

- **Ngày có thể không cân bằng:** K-Means tối ưu độ gọn địa lý, không ép mỗi ngày có cùng số điểm hay cùng tổng thời gian.
- **Có thể xuất hiện cụm rỗng:** code giữ lại centroid cũ và bỏ qua cụm rỗng khi dựng itinerary, nên số ngày có điểm thực tế có thể ít hơn số ngày yêu cầu.
- **Không có mạng lưới đường bộ:** Haversine là đường chim bay; núi, sông, cầu, phà và chất lượng đường có thể làm thời gian thực tế rất khác.
- **Không có ràng buộc thời gian:** mô hình chưa xét giờ mở cửa, thời lượng tham quan, giờ nghỉ, điểm xuất phát/kết thúc, khách sạn hay giới hạn lái xe mỗi ngày.
- **Không cân bằng sở thích và khoảng cách:** sau Stage 2, Stage 3 chủ yếu dùng tọa độ; nó chưa tối ưu đồng thời điểm recommendation, độ ưu tiên “must-see” và chi phí di chuyển.
- **Heuristic không có bảo đảm tối ưu:** cả cách phân ngày lẫn xếp tuyến đều có thể được cải thiện bằng mô hình ràng buộc, dữ liệu thời gian đường bộ hoặc solver chuyên dụng.

### Self-check

Trước khi xem lời giải, hãy tự trả lời:
1. Vì sao trong notebook này chọn `K = số ngày`? Đây là định nghĩa toán học hay một giả định mô hình hóa?
2. Hai bước Assign và Update của K-Means thay đổi `clusters` và `centroids` như thế nào?
3. Vì sao K-Means++ dùng `D(x)²` thay vì chọn đều mọi điểm? Seed có tác dụng gì và không có tác dụng gì?
4. Euclidean trên `(lat, lng)`, Haversine và khoảng cách lái xe khác nhau ở đâu?
5. Vì sao chạy Nearest Neighbor từ nhiều điểm bắt đầu thường tốt hơn chỉ chạy một lần?
6. Route trong code có bao nhiêu chặng với `n` điểm? Vì sao nó không phải một tour TSP khép kín?
7. Tại sao một cụm địa lý “gọn” vẫn có thể tạo ra một ngày quá tải?
8. Muốn đưa itinerary vào thực tế, bạn sẽ bổ sung ba loại dữ liệu hoặc ràng buộc nào trước tiên?

Sau khi chạy được notebook, hãy kiểm tra các bất biến sau:
- mỗi điểm đã chọn xuất hiện đúng một lần trong toàn bộ itinerary;
- không ngày nào chứa index ngoài danh sách đã chọn;
- `day_total_km` xấp xỉ tổng các phần tử trong `legs_km`;
- số ngày có điểm không vượt quá `min(n_days, số điểm đã chọn)`;
- với cùng dữ liệu và cùng seed, kết quả phân cụm có thể tái lập.